# Model training and selection

In [1]:
from pathlib import Path
import sys, json
ROOT = Path.cwd() if (Path.cwd() / 'src').exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
import pandas as pd
import numpy as np
from IPython.display import display, Image, Markdown
from src.data_loader import load_partition, predictors, fingerprints
def report(name):
    display(Markdown((ROOT / 'reports' / name).read_text(encoding='utf-8')))
def figure(name):
    display(Image(filename=str(ROOT / 'figures' / name)))


The expensive training experiment was executed with `python -m src.train`. This notebook inspects its reproducible configuration and saved results; rerun that command to retrain. The primary selection metric is average precision.

In [2]:
metadata=json.loads((ROOT/'models/metadata.json').read_text())
display(metadata)
for name in ['logistic_regression','random_forest','xgboost']:
    cv=pd.read_csv(ROOT/f'reports/tuning_{name}.csv')
    display(cv[['params','mean_train_score','mean_test_score','std_test_score','rank_test_score']])

{'model': 'XGBoost',
 'threshold': 0.49,
 'features': ['dur',
  'proto',
  'service',
  'state',
  'spkts',
  'dpkts',
  'sbytes',
  'dbytes',
  'rate',
  'sttl',
  'dttl',
  'sload',
  'dload',
  'sloss',
  'dloss',
  'sinpkt',
  'dinpkt',
  'sjit',
  'djit',
  'swin',
  'stcpb',
  'dtcpb',
  'dwin',
  'tcprtt',
  'synack',
  'ackdat',
  'smean',
  'dmean',
  'trans_depth',
  'response_body_len',
  'ct_srv_src',
  'ct_state_ttl',
  'ct_dst_ltm',
  'ct_src_dport_ltm',
  'ct_dst_sport_ltm',
  'ct_dst_src_ltm',
  'is_ftp_login',
  'ct_ftp_cmd',
  'ct_flw_http_mthd',
  'ct_src_ltm',
  'ct_srv_dst',
  'is_sm_ips_ports'],
 'created_utc': '2026-09-15T10:19:02.738623+00:00',
 'seed': 42,
 'selection': 'Highest validation average precision; validation-only threshold',
 'constraint_met': False,
 'search_rows': 30000,
 'iterations': 4,
 'cv_folds': 3,
 'timing': {'Logistic Regression': {'training_seconds': 23.665719399999944,
   'best_params': {'model__class_weight': 'balanced', 'model__C': 10}}

,params,mean_train_score,mean_test_score,std_test_score,rank_test_score
0,"{'model__class_weight': 'balanced', 'model__C'...",0.942594,0.941231,0.001846,4
1,"{'model__class_weight': 'balanced', 'model__C'...",0.958388,0.956936,0.000485,2
2,"{'model__class_weight': None, 'model__C': 0.01}",0.942925,0.941559,0.001802,3
3,"{'model__class_weight': 'balanced', 'model__C'...",0.960583,0.958574,0.001544,1


,params,mean_train_score,mean_test_score,std_test_score,rank_test_score
0,"{'model__min_samples_leaf': 1, 'model__max_dep...",0.993180,0.980592,0.000644,3
1,"{'model__min_samples_leaf': 3, 'model__max_dep...",0.990981,0.980267,0.000641,4
2,"{'model__min_samples_leaf': 8, 'model__max_dep...",0.993382,0.980921,0.000475,1
3,"{'model__min_samples_leaf': 8, 'model__max_dep...",0.993133,0.980862,0.000465,2


,params,mean_train_score,mean_test_score,std_test_score,rank_test_score
0,"{'model__subsample': 0.8, 'model__scale_pos_we...",0.995341,0.981614,0.000451,1
1,"{'model__subsample': 0.8, 'model__scale_pos_we...",0.995030,0.981330,0.000545,2
2,"{'model__subsample': 0.8, 'model__scale_pos_we...",0.980397,0.977595,0.000304,4
3,"{'model__subsample': 0.8, 'model__scale_pos_we...",0.994987,0.981157,0.000318,3


In [3]:
validation=pd.read_csv(ROOT/'reports/validation_comparison.csv')
display(validation)
assert validation.sort_values('pr_auc',ascending=False).iloc[0]['model']==metadata['model']

,model,threshold,constraint_met,cv_pr_auc,n,accuracy,precision,recall,f1,roc_auc,pr_auc,fpr,fnr,tn,fp,fn,tp
0,Logistic Regression,0.17,False,0.958574,20163,0.902445,0.834681,0.997457,0.908838,0.968426,0.961143,0.187942,0.002543,8391,1942,25,9805
1,Random Forest,0.52,False,0.980921,20163,0.932599,0.918073,0.946185,0.931917,0.985560,0.982917,0.080325,0.053815,9503,830,529,9301
2,XGBoost,0.49,False,0.981614,20163,0.934831,0.917287,0.952187,0.934412,0.986236,0.984289,0.081680,0.047813,9489,844,470,9360


Interpretation: four candidates and three folds per family provide a bounded comparison. Training versus CV scores help inspect overfitting, but a small search is not proof of global optimality. Validation may itself be optimistic because it supports model and threshold selection.